<a href="https://colab.research.google.com/github/Tri27sen/chatbot_cbt/blob/main/Upload_and__test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sat Feb  8 13:39:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "trisha2710/chat-doctor-merged"

# Load the model with optimizations
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,   # Use lower precision (reduces memory usage)
    low_cpu_mem_usage=True,      # Optimize CPU memory usage
    device_map="auto"            # Use GPU if available
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create chatbot pipeline
chatbot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"  # Uses GPU if available, otherwise CPU
)

# Run inference
response = chatbot("What are the symptoms of depression?", max_length=250)
print(response[0]["generated_text"])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.51M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


What are the symptoms of depression? How do I know if I have depression? I've been feeling sad for the last three weeks. I've been crying a lot and I'm not interested in anything. I feel like I'm not worth anything. I feel like I'm annoying and people are always annoyed with me. I feel like I'm worthless and I'm not good at anything. I feel like I'm not loved. I feel like I'm alone. I feel like I'm not worth knowing. I feel like I'm not worth anything. I feel like I'm not good at anything. I feel like I'm not worth knowing. I feel like I'm not loved. I feel like I'm alone. I feel like I'm worthless. I feel like I'm not worth anything. I feel like I'm not worth anything. I feel like I'm not worth anything. I feel like I'm not worth anything. I feel like I'm not worth anything. I feel like I'm not worth anything. I feel like I'm not worth anything. I feel like I'm not worth


In [8]:



# Ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or '[PAD]'  # Use EOS if available, else add new PAD token
    tokenizer.add_special_tokens({'pad_token': tokenizer.pad_token})

    # Now resize token embeddings
    model.resize_token_embeddings(len(tokenizer))

In [7]:
import torch
torch.cuda.empty_cache()


In [5]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32001, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,), eps=1e-0

**1️⃣ Emotional Well-being**

In [9]:
messages = [
    {
        "role": "user",
        "content": " I feel unworthy of love an affection "
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True,
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST]  I feel unworthy of love an affection  [/INST] Feeling unworthy of love and affection can be a difficult and painful emotion to deal with. It's important to remember that everyone deserves to be loved and cared for, regardless of their past experiences or current circumstances.

It's possible that you may have had negative experiences in the past that have led you to believe that you are unworthy of love. However, it's important to recognize that these thoughts may not be reflective of reality. It's also possible that you may be struggling with low self-esteem or other mental health issues that are contributing to these feelings.

One way to address these feelings is to challenge them. Ask yourself if there is any evidence to support your belief that you are unworthy of love. Chances are, you won't be able to come up with any examples. It's also important to practice self-compassion and kindness towards yourself. Treat yourself the way you would treat a good friend who is going 

**2️⃣Sleep & Relaxation**

In [10]:
messages = [
    {
        "role": "user",
        "content": " I feel guilty when i take a break"
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True,
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST]  I feel guilty when i take a break [/INST] It sounds like you may be struggling with feelings of guilt when taking time for yourself. It's important to remember that self-care and taking breaks are essential for maintaining mental health and well-being. Here are a few strategies that may help:

1. Reframe your thoughts: Instead of feeling guilty, try to see taking breaks as a necessary part of self-care. Remind yourself that it's okay to prioritize your own needs and that taking a break doesn't mean you're neglecting your responsibilities.
2. Practice mindfulness: When you start to feel guilty about taking a break, try taking a few deep breaths and focusing on the present moment. This can help you become more aware of your thoughts and feelings without getting caught up in them.
3. Set boundaries: If you find that you're constantly feeling guilty about taking breaks, it may be helpful to set some boundaries for yourself. For example, you could try setting a specific amount of ti

**3️⃣ Coping Mechanisms & Stress Management**

In [11]:
messages = [
    {
        "role": "user",
        "content": " Recommend some yoga for stress"
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True,
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST]  Recommend some yoga for stress [/INST] If you're looking for yoga styles that can help with stress, here are a few options you may want to try:

1. Hatha Yoga: This style of yoga is gentle and slow-paced, making it perfect for beginners or those looking for a relaxing practice. The focus is on holding poses for longer periods of time, which can help calm the mind and reduce stress.
2. Vinyasa Yoga: This style of yoga is faster-paced and focuses on flowing from one pose to the next. The movement can help release tension in the body and the breath work can help calm the mind.
3. Yin Yoga: This style of yoga targets the deep connective tissues, joints and fascia in the body. The long-held passive stretches can help reduce stress and anxiety by promoting relaxation and releasing tension in the body.
4. Restorative Yoga: This style of yoga is designed to help you relax and restore your body and mind. Poses are held for longer periods of time and props are used to support the body, a

**4️⃣ Cognitive Behavioral Therapy (CBT)-Based Questions**

In [12]:
messages = [
    {
        "role": "user",
        "content": "What are some positive affirmations I can use daily?"
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True,
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] What are some positive affirmations I can use daily? [/INST] Here are some positive affirmations you can use daily:

1. I am worthy of love and respect.
2. I am capable of achieving my goals.
3. I am confident in myself and my abilities.
4. I am deserving of happiness and success.
5. I am grateful for the blessings in my life.
6. I am at peace with myself and others.
7. I am worthy of forgiveness and compassion.
8. I am strong enough to overcome any obstacle.
9. I am deserving of a fulfilling and meaningful life.
10. I am worthy of self-love and self-care.


**5️⃣ Social & Relationship Anxiety**

In [13]:
messages = [
    {
        "role": "user",
        "content": "How can I handle rejection better?"
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True,
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] How can I handle rejection better? [/INST] Rejection can be difficult to handle, but there are several things you can try to make it easier:

1. Practice self-compassion: Treat yourself the way you would treat a good friend who has just experienced rejection. Be kind, understanding, and forgiving.
2. Reframe the situation: Instead of focusing on the rejection, try to see the positive aspects of the situation. For example, if you were rejected because the other person met someone else they're more compatible with, that doesn't reflect anything about you.
3. Don't take it personally: Remember that rejection is not a reflection of your worth as a person. It's often about the other person and their needs, not about you.
4. Learn from it: Use rejection as an opportunity to learn and grow. Ask yourself what you could have done differently, and use that information to improve yourself in the future.
5. Take care of yourself: Make sure you're getting enough sleep, eating well, and exerc

**Depression**

In [20]:
messages = [
    {
        "role": "user",
        "content": "How can I find motivation when I feel down?"
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True,
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [21]:
print(text)

[INST] How can I find motivation when I feel down? [/INST] Finding motivation when you're feeling down can be challenging, but there are several things you can try to get started:

1. Identify your goals: Take some time to think about what you want to achieve and why it's important to you. Having clear goals can help give you something to focus on and work towards.

2. Break it down into smaller steps: If your goal seems too big or overwhelming, break it down into smaller, more manageable steps. This can help make the task feel less daunting and more achievable.

3. Find an accountability partner: Having someone to hold you accountable can be a great motivator. Find a friend, family member, or colleague who can help keep you on track and offer support when you need it.

4. Reward yourself: Set up a reward system for yourself. When you complete a task or make progress towards your goal, treat yourself to something that you enjoy. This can help give you something to look forward to and p

**6️⃣ Low Mood**

In [18]:
messages = [
    {
        "role": "user",
        "content": "What are some small steps I can take to feel better??"
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True,
                   truncation=True, max_length=512).to("cuda")


outputs = model.generate(**inputs, max_length=250, num_return_sequences=1, num_beams=5, early_stopping=False, repetition_penalty=2.2)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [19]:
print(text)

[INST] What are some small steps I can take to feel better?? [/INST] There are many small steps you can take to improve your mood and overall well-being. Here are a few ideas:

1. Get enough sleep: Aim for 7-9 hours of quality sleep each night.
2. Exercise regularly: Try to get at least 30 minutes of moderate exercise, such as brisk walking or cycling, most days of the week.
3. Eat a healthy diet: Focus on whole, nutrient-dense foods like fruits, vegetables, lean proteins, and whole grains.
4. Stay hydrated: Drink plenty of water throughout the day.
5. Connect with others: Spend time with friends and family members who make you feel happy and supported.
6. Practice self-care: Take time for activities that help you relax and recharge, such as reading, taking a bath, or practicing mindfulness.
7. Keep a gratitude journal: Write down three things you're grateful for each day.
8. Seek professional help if needed: If you're struggling with symptoms of depression or anxiety,
